In [1]:
import pandas as pd
import numpy as np
import joblib
import json
import warnings
warnings.filterwarnings('ignore')

In [2]:
print("Loading models...")

# Load data 
X = pd.read_csv('../data/processed/X_features.csv')
feature_columns = joblib.load('../data/models/feature_columns.pkl')
scaler = joblib.load('../data/models/scaler.pkl')

# Load clustering models
kmeans = joblib.load('../data/models/kmeans_model.pkl')
anomaly_threshold = joblib.load('../data/models/anomaly_threshold.pkl')

# Load classification models
dt_model = joblib.load('../data/models/decision_tree.pkl')
nb_model = joblib.load('../data/models/naive_bayes.pkl')
rf_model = joblib.load('../data/models/random_forest.pkl')

# Load anomaly detection models
iso_forest = joblib.load('../data/models/isolation_forest.pkl')
combined_threshold = joblib.load('../data/models/combined_threshold.pkl')

print(" All models loaded successfully!")

Loading models...
 All models loaded successfully!


In [3]:
def predict_household_anomaly(consumption_data):
    """
    Complete prediction pipeline for a single household
    
    Parameters:
    consumption_data: array of 96 values (24 hours * 15-min intervals)
    
    Returns:
    dict with prediction results
    """
    import pandas as pd
    import numpy as np
    
    # Step 1: Extract features from raw consumption data
    def extract_features_from_raw(raw_data):
        """Extract the same features used in training"""
        if len(raw_data) != 96:
            raise ValueError(f"Expected 96 readings, got {len(raw_data)}")
        
        features = {}
        
        # Basic statistics
        features['total_daily'] = np.sum(raw_data)
        features['avg_consumption'] = np.mean(raw_data)
        features['std_consumption'] = np.std(raw_data)
        features['min_consumption'] = np.min(raw_data)
        features['max_consumption'] = np.max(raw_data)
        features['median_consumption'] = np.median(raw_data)
        features['consumption_range'] = np.max(raw_data) - np.min(raw_data)
        features['cv'] = features['std_consumption'] / (features['avg_consumption'] + 0.001)
        
        # Time-based features (96 readings = 24 hours, 4 per hour)
        morning_idx = range(24, 36)    # 6 AM - 9 AM (24 to 36 readings)
        evening_idx = range(60, 72)    # 3 PM - 6 PM (60 to 72 readings)
        night_idx = list(range(0, 20)) + list(range(80, 96))  # 11 PM - 5 AM
        
        features['morning_peak'] = np.max(raw_data[morning_idx]) if len(raw_data[morning_idx]) > 0 else 0
        features['evening_peak'] = np.max(raw_data[evening_idx]) if len(raw_data[evening_idx]) > 0 else 0
        features['night_consumption'] = np.mean(raw_data[night_idx]) if len(raw_data[night_idx]) > 0 else 0
        features['night_ratio'] = features['night_consumption'] / (features['avg_consumption'] + 0.001)
        
        # Zero detection
        features['zero_intervals'] = np.sum(raw_data < 0.1)
        features['zero_proportion'] = features['zero_intervals'] / 96
        
        # Peak hour
        peak_idx = np.argmax(raw_data)
        features['peak_hour'] = (peak_idx / 4) if peak_idx < len(raw_data) else 0
        
        # Ramp rates
        morning_ramp_idx = range(20, 32)  # 5 AM - 8 AM
        evening_ramp_idx = range(80, 92)  # 8 PM - 11 PM
        
        if len(raw_data[morning_ramp_idx]) > 0:
            features['morning_ramp'] = np.max(raw_data[morning_ramp_idx]) - np.min(raw_data[morning_ramp_idx])
        else:
            features['morning_ramp'] = 0
            
        if len(raw_data[evening_ramp_idx]) > 0:
            features['evening_ramp'] = np.max(raw_data[evening_ramp_idx]) - np.min(raw_data[evening_ramp_idx])
        else:
            features['evening_ramp'] = 0
        
        # Load factor
        features['load_factor'] = features['avg_consumption'] / (features['max_consumption'] + 0.001)
        
        # Rolling features (simplified for single day)
        features['rolling_7day_mean'] = features['total_daily']
        features['rolling_7day_std'] = features['std_consumption']
        features['rolling_7day_max'] = features['max_consumption']
        features['zscore_7day'] = 0
        
        # Temporal features (default values for single prediction)
        # In production, you'd get actual date/time
        features['day_of_week'] = pd.Timestamp.now().dayofweek
        features['is_weekend'] = 1 if features['day_of_week'] >= 5 else 0
        features['is_monday'] = 1 if features['day_of_week'] == 0 else 0
        features['is_friday'] = 1 if features['day_of_week'] == 4 else 0
        features['month'] = pd.Timestamp.now().month
        features['season'] = 0 if features['month'] in [12,1,2] else 1 if features['month'] in [3,4,5] else 2 if features['month'] in [6,7,8] else 3
        
        return features
    
    # Extract features
    raw_features = extract_features_from_raw(consumption_data)
    features_df = pd.DataFrame([raw_features])
    
    # Ensure all required feature columns are present
    required_cols = joblib.load('../data/models/feature_columns.pkl')
    for col in required_cols:
        if col not in features_df.columns:
            features_df[col] = 0
    
    features_df = features_df[required_cols]
    
    # Scale features
    features_scaled = scaler.transform(features_df)
    
    # Step 2: Get cluster assignment
    cluster = int(kmeans.predict(features_scaled)[0])
    
    # Step 3: Calculate distance-based anomaly
    from scipy.spatial.distance import cdist
    distance = np.min(cdist(features_scaled, kmeans.cluster_centers_))
    distance_anomaly = distance > anomaly_threshold
    
    # Step 4: Classification model predictions
    dt_pred = int(dt_model.predict(features_scaled)[0])
    nb_pred = int(nb_model.predict(features_scaled)[0])
    rf_pred = int(rf_model.predict(features_scaled)[0])
    rf_proba = float(rf_model.predict_proba(features_scaled)[0][1])
    
    # Step 5: Isolation Forest
    iso_pred = int(iso_forest.predict(features_scaled)[0] == -1)
    
    # Step 6: Ensemble decision (majority voting)
    votes = sum([distance_anomaly, dt_pred, nb_pred, rf_pred, iso_pred])
    is_anomaly = votes >= 3  # Majority of 5 models
    confidence = votes / 5.0
    
    # Step 7: Determine anomaly type (if applicable)
    anomaly_type = "Normal"
    if is_anomaly:
        # Simple heuristic for anomaly type based on features
        if raw_features['night_ratio'] < 0.1:
            anomaly_type = "Night Zeroing (Possible Meter Tampering)"
        elif raw_features['load_factor'] < 0.3:
            anomaly_type = "Peak Clipping (Meter Bypass Suspected)"
        elif raw_features['zero_proportion'] > 0.2:
            anomaly_type = "Extended Zero Periods (Disconnection Suspected)"
        elif raw_features['avg_consumption'] > raw_features['rolling_7day_mean'] * 1.5:
            anomaly_type = "Sudden Spike (Illegal Connection Suspected)"
        else:
            anomaly_type = "Unusual Pattern (Further Investigation Required)"
    
    # Return comprehensive result
    return {
        'is_anomaly': bool(is_anomaly),
        'confidence': confidence,
        'anomaly_type': anomaly_type,
        'anomaly_score': float(1 - confidence),
        'cluster': cluster,
        'models': {
            'kmeans_distance': bool(distance_anomaly),
            'decision_tree': bool(dt_pred),
            'naive_bayes': bool(nb_pred),
            'random_forest': bool(rf_pred),
            'isolation_forest': bool(iso_pred)
        },
        'voting': f"{votes}/5 models flagged as anomaly",
        'rf_probability': rf_proba,
        'distance_to_center': float(distance)
    }

# Test the function with sample data
print("\nTesting prediction function with random data...")
test_data = np.random.rand(96) * 2  # Random consumption between 0-2 kWh
result = predict_household_anomaly(test_data)
print(f"Prediction result: {'ANOMALY' if result['is_anomaly'] else 'NORMAL'}")
print(f"Confidence: {result['confidence']:.2%}")
print(f"Votes: {result['voting']}")



Testing prediction function with random data...
Prediction result: NORMAL
Confidence: 20.00%
Votes: 1/5 models flagged as anomaly


In [4]:
def batch_predict_households(consumption_matrix):
    """
    Predict for multiple households
    
    Parameters:
    consumption_matrix: 2D array (n_households × 96)
    
    Returns:
    DataFrame with predictions
    """
    results = []
    for i, consumption in enumerate(consumption_matrix):
        try:
            pred = predict_household_anomaly(consumption)
            pred['household_id'] = i
            results.append(pred)
        except Exception as e:
            print(f"Error processing household {i}: {e}")
            results.append({
                'household_id': i,
                'is_anomaly': None,
                'error': str(e)
            })
    
    return pd.DataFrame(results)

print(" Batch prediction function created!")

 Batch prediction function created!


In [5]:
model_package = {
    'scaler': scaler,
    'kmeans': kmeans,
    'anomaly_threshold': anomaly_threshold,
    'decision_tree': dt_model,
    'naive_bayes': nb_model,
    'random_forest': rf_model,
    'isolation_forest': iso_forest,
    'combined_threshold': combined_threshold,
    'feature_columns': feature_columns,
    'predict_function': predict_household_anomaly
}

# Save as single file for easy loading in web app
joblib.dump(model_package, '../data/models/model_package.pkl')
print(" Model package saved to models/model_package.pkl")

# Also save as JSON metadata for web app
metadata = {
    'feature_count': len(feature_columns),
    'feature_names': feature_columns,
    'n_clusters': kmeans.n_clusters,
    'anomaly_threshold': float(anomaly_threshold),
    'models': {
        'decision_tree': 'DecisionTreeClassifier',
        'naive_bayes': 'GaussianNB',
        'random_forest': 'RandomForestClassifier',
        'isolation_forest': 'IsolationForest'
    }
}

with open('../data/models/model_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=4)

print(" Model metadata saved!")

 Model package saved to models/model_package.pkl
 Model metadata saved!


In [6]:
sample_households = []
for i in range(10):
    # Normal pattern (baseline)
    base_pattern = np.random.rand(96) * 1.5
    base_pattern[24:36] = base_pattern[24:36] * 1.5  # Morning peak
    base_pattern[60:72] = base_pattern[60:72] * 2    # Evening peak
    base_pattern[80:96] = base_pattern[80:96] * 0.3   # Night low
    sample_households.append(base_pattern)

# Add some anomaly patterns
for i in range(5):
    anomaly_pattern = np.random.rand(96) * 2
    if i % 2 == 0:
        # Peak clipping
        anomaly_pattern[anomaly_pattern > 1.2] = 1.2
    else:
        # Night zeroing
        anomaly_pattern[80:96] = 0
    sample_households.append(anomaly_pattern)

# Convert to DataFrame for CSV
sample_df = pd.DataFrame(sample_households)
sample_df.columns = [f'interval_{i+1}' for i in range(96)]
sample_df['household_id'] = [f'HH_{i+1:03d}' for i in range(len(sample_households))]

# Add predictions for verification
predictions = []
for _, row in sample_df.iterrows():
    consumption = row[[f'interval_{i+1}' for i in range(96)]].values
    pred = predict_household_anomaly(consumption)
    predictions.append(pred['is_anomaly'])

sample_df['is_anomaly_predicted'] = predictions
sample_df.to_csv('../data/processed/sample_predictions.csv', index=False)

print(" Sample predictions CSV saved!")
print(f"Sample file: {len(sample_households)} households ({(sample_df['is_anomaly_predicted'].sum())} anomalies)")

 Sample predictions CSV saved!
Sample file: 15 households (8 anomalies)


In [7]:
print("\n" + "="*60)
print("MODEL EXPORT COMPLETE")
print("="*60)
print("""
Files exported for web application:
├── data/models/
│   ├── model_package.pkl          # All models in one file
│   ├── model_metadata.json        # Model information
│   ├── scaler.pkl                 # Feature scaler
│   ├── kmeans_model.pkl           # K-Means clustering
│   ├── isolation_forest.pkl       # Isolation Forest
│   ├── decision_tree.pkl          # Decision Tree
│   ├── naive_bayes.pkl            # Naive Bayes
│   ├── random_forest.pkl          # Random Forest
│   └── feature_columns.pkl        # Feature names
│
├── data/processed/
│   └── sample_predictions.csv     # Sample data for testing
│
└── notebooks/                     # All 7 notebooks complete

Ready for web application development!
""")

print("\n ALL NOTEBOOKS COMPLETE! ")


MODEL EXPORT COMPLETE

Files exported for web application:
├── data/models/
│   ├── model_package.pkl          # All models in one file
│   ├── model_metadata.json        # Model information
│   ├── scaler.pkl                 # Feature scaler
│   ├── kmeans_model.pkl           # K-Means clustering
│   ├── isolation_forest.pkl       # Isolation Forest
│   ├── decision_tree.pkl          # Decision Tree
│   ├── naive_bayes.pkl            # Naive Bayes
│   ├── random_forest.pkl          # Random Forest
│   └── feature_columns.pkl        # Feature names
│
├── data/processed/
│   └── sample_predictions.csv     # Sample data for testing
│
└── notebooks/                     # All 7 notebooks complete

Ready for web application development!


 ALL NOTEBOOKS COMPLETE! 
